In [ ]:
%load_ext autoreload
%autoreload all

In [ ]:
# Install the CVAT's fork of Datumaro.
# It has an improved and more flexible YOLO Ultralytics exporter.
# It requires Rust!

# To install Rust, either:
# curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh
# or
# sudo apt install rustup

# To install CVAT's Datumaro fork:
# pip install "datumaro[default] @ git+https://github.com/cvat-ai/datumaro"

In [ ]:
# External imports
from ultralytics import YOLO
from datumaro.components.annotation import AnnotationType

# Local imports
from dataset_builder import DatumaroDatasetBuilder
from convert_utils import (
    get_frame_chunks_df,
    load_masks,
    load_annotations,
    load_categories,
    load_errors_df,
    extract_error_frames,
)
from blob_filter_rules import MinAreaRule, MinSizeRule
from anomaly_rules import create_anomaly_rules
from configuration import Config
from multi_dataset_builder import find_masks, find_annot
from configuration import ClassifierConfig

### Original frame number:

- **The annotations file:** uses the original (raw) frame number.

### Extracted frame index:
- **The error file:** Uses the extracted frame index (starts at 0).
- **The filenames of the extracted frames:** May start at 0001.jpg, or may start at a random number, due to different extraction methods for different projects. If possible, you should use the index of the extracted frames, not the file name.
- **The masks dict:** Uses the extracted frame index.

In [ ]:
classifier = YOLO(ClassifierConfig.model_weights_path)

In [ ]:
# Blobs will be pre-filtered according to these rules
blob_filter_rules = [
    MinAreaRule(Config.area_threshold),
    MinSizeRule(Config.min_size_threshold),
]

# Anomalies across time in blob properties will be detected using these rules
anomaly_rules = create_anomaly_rules(Config.anomaly_rules)

In [ ]:
# Select which observation to work on
obs_id_object, images_path = list(Config.obsId_to_folder_map.items())[0]

In [ ]:
masks_filepath = find_masks(Config.masks_path, obs_id_object)
annot_filepath = find_annot(Config.annot_path, obs_id_object)

masks = load_masks(masks_filepath)
annotations_df = load_annotations(annot_filepath)
label_categories = load_categories(annotations_df)
chunked_df = get_frame_chunks_df(annotations_df)

obs_id_str = obs_id_object.to_str()
video_errors_df = load_errors_df(Config.errors_csv_filepath, obs_id_str)
all_error_frames = extract_error_frames(video_errors_df)

In [ ]:
annotations_df

In [ ]:
annotations_df.ObjType.unique()

In [ ]:
label_categories[AnnotationType.label]

In [ ]:
chunked_df

In [ ]:
video_errors_df.error_type.unique()

In [ ]:
all_error_frames

In [ ]:
builder = DatumaroDatasetBuilder(
    obs_id=obs_id_str,
    error_frames=all_error_frames,
    masks=masks,
    chunked_df=chunked_df,
    label_categories=label_categories,
    classifier=classifier,
    blob_rules=blob_filter_rules,
    anomaly_rules=anomaly_rules,
    target_class="correct_fish_mask",
    images_path=images_path,
    classifier_conf=ClassifierConfig.classifier_conf,
    start_frame=Config.start_frame,
    max_frames=Config.max_frames,
    filename_num_zeros=Config.number_of_zeros,
    export_root_path=Config.output_path / obs_id_str,
    window_size=Config.window_size,
    verbose=False,
    notebook_debug=False,
)
dataset = builder.build()

In [ ]:
dataset

In [ ]:
export_root_path = Config.output_path / obs_id_str

In [ ]:
cvat_out = export_root_path / "dataset_cvat"
dataset.export(str(cvat_out), format="cvat")
print(f"Annotations exported to '{cvat_out}'")

In [ ]:
yolo_out = export_root_path / "dataset_yolo"
dataset.export(
    str(yolo_out),
    format="yolo_ultralytics_detection",
    add_path_prefix=False,
    save_media=True,
)
print(f"Annotations exported to '{yolo_out}'")